# 06 - Interactive Visual Analytics with Folium
**IBM Applied Data Science Capstone - SpaceX Falcon 9 First-Stage Landing Prediction**

Author: Pranav Vyankatesh Jagdale

**Goal:** map the launch sites, mark every launch as success / failure, and measure distances to nearby features.

> Run every cell in order. Keep your own outputs - they feed your final slides.

In [ ]:
!pip install folium -q

In [ ]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")
print(spacex_df.columns.tolist())
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby('Launch Site', as_index=False).first()[['Launch Site', 'Lat', 'Long']]
launch_sites_df

## 1. Mark every launch site

In [ ]:
site_map = folium.Map(location=[29.559684888503615, -95.0830971930759], zoom_start=5)
for _, r in launch_sites_df.iterrows():
    coord = [r['Lat'], r['Long']]
    folium.Circle(coord, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(r['Launch Site'])).add_to(site_map)
    folium.map.Marker(coord, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html=f'<div style="font-size:12px;color:#d35400;"><b>{r["Launch Site"]}</b></div>')).add_to(site_map)
site_map

## 2. Success / failure markers

In [ ]:
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)
for _, r in spacex_df.iterrows():
    marker_cluster.add_child(folium.Marker([r['Lat'], r['Long']],
        icon=folium.Icon(color='white', icon_color=r['marker_color'])))
site_map

## 3. Distances to nearby features
Add a mouse-position readout, hover over the coastline, a railway, a highway and the nearest city, and note the coordinates.

In [ ]:
MousePosition(position='topright', separator=' Long: ', empty_string='NaN', lng_first=False,
              num_digits=20, prefix='Lat:', lat_formatter="function(num) {return L.Util.formatNum(num, 5);};",
              lng_formatter="function(num) {return L.Util.formatNum(num, 5);};").add_to(site_map)
site_map

In [ ]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# Pick one launch site and enter coordinates YOU read from the map above.
site_name = launch_sites_df.iloc[0]['Launch Site']
site_lat, site_lon = launch_sites_df.iloc[0]['Lat'], launch_sites_df.iloc[0]['Long']

coast_lat, coast_lon = None, None   # <- fill in from the mouse-position readout

assert coast_lat is not None, "Enter the coastline coordinates you read from the map"
d_coast = calculate_distance(site_lat, site_lon, coast_lat, coast_lon)
print(f"{site_name} -> coastline: {d_coast:.2f} km")

folium.Marker([coast_lat, coast_lon],
    icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                 html=f'<div style="font-size:12px;color:#1f618d;"><b>{d_coast:.2f} KM</b></div>')).add_to(site_map)
folium.PolyLine([[site_lat, site_lon], [coast_lat, coast_lon]], weight=1).add_to(site_map)
site_map

Repeat the same pattern for a railway, a highway and the nearest city. Then save the map and keep a few screenshots for your slides.

In [ ]:
site_map.save('my_spacex_folium_map.html')

## Write your own findings here
Are launch sites near the coast? Near railways and highways? Far from cities? Why do you think that is?